# Crop Images

This notebook allows to crop images to remove excess background (white background, i.e., the color that appears most frequently).
This is used to crop the images, written on a tablet.

## Libraries

In [1]:
from PIL import Image
from collections import Counter
import glob
import torch
import numpy as np


## Code

First load all images from the desired folder

In [ ]:
image_list = []

for filename in glob.glob('images/class_diagram5/*.jpg'):
    im = Image.open(filename)
    image_list.append(im)

Count the number of images

In [8]:
len(image_list)

54

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


To crop the image, we count first the color with the most occurence among all pixels since the background is not entirely white ( #fffff ). This color (and #ffffff ) are used as reference to crop the images.

In [ ]:
# Specify path to save the new images
output_dest_path = "images/class_diagram5"

In [12]:
for idx, image in enumerate(image_list):
    
    # Convert to RGB
    if image.mode != 'RGB':
        image = image.convert('RGB')
    
    # Convert PIL image to numpy array
    img_array = np.array(image)
    
    # Find most common color (still on CPU, as Counter is efficient for this)
    pixels = img_array.reshape(-1, 3)
    pixels_tuple = [tuple(p) for p in pixels]
    color_counts = Counter(pixels_tuple)
    most_common_color = color_counts.most_common(1)[0][0]
    
    # Move to GPU for mask computation
    img_tensor = torch.from_numpy(img_array).to(device).float()  # Convert to float for distance calculation
    
    # Create background mask for most common color (exact match)
    bg_color1 = torch.tensor(most_common_color, device=device, dtype=torch.float32)
    mask1 = (img_tensor == bg_color1).all(dim=2)
    
    # Create background mask for near-white colors (within tolerance of 10)
    white_color = torch.tensor([255.0, 255.0, 255.0], device=device)
    # Calculate Euclidean distance to white for each pixel
    distance_to_white = torch.norm(img_tensor - white_color, dim=2)
    mask2 = distance_to_white <= 10.0  # Tolerance of 10
    
    # Combine masks
    non_bg_mask = ~(mask1 | mask2)
    
    # Find bounding box
    if non_bg_mask.any():
        rows = non_bg_mask.any(dim=1)
        cols = non_bg_mask.any(dim=0)
        
        top = rows.nonzero()[0].item()
        bottom = rows.nonzero()[-1].item() + 1
        left = cols.nonzero()[0].item()
        right = cols.nonzero()[-1].item() + 1
        
        # Add 10-pixel padding to all sides (clamp to image boundaries)
        height, width = img_array.shape[:2]
        padding = 10
        top = max(0, top - padding)
        bottom = min(height, bottom + padding)
        left = max(0, left - padding)
        right = min(width, right + padding)
        
        # Crop and save
        cropped_array = img_array[top:bottom, left:right]
        cropped_img = Image.fromarray(cropped_array)
        output_path = f"{output_dest_path}/image_{idx:04d}.png"
        cropped_img.save(output_path)
        print(f"Cropped image {idx} to: {left}, {top}, {right}, {bottom}")
    else:
        print(f"Image {idx}: No non-background pixels found, skipping crop")

Cropped image 0 to: 155, 90, 755, 473
Cropped image 1 to: 55, 23, 392, 596
Cropped image 2 to: 104, 37, 362, 548
Cropped image 3 to: 13, 27, 433, 667
Cropped image 4 to: 56, 13, 382, 571
Cropped image 5 to: 36, 30, 513, 774
Cropped image 6 to: 65, 19, 386, 635
Cropped image 7 to: 41, 80, 508, 239
Cropped image 8 to: 71, 55, 298, 420
Cropped image 9 to: 41, 12, 298, 571
Cropped image 10 to: 95, 20, 357, 780
Cropped image 11 to: 110, 100, 488, 410
Cropped image 12 to: 155, 11, 481, 1076
Cropped image 13 to: 126, 15, 718, 477
Cropped image 14 to: 96, 66, 506, 366
Cropped image 15 to: 99, 78, 684, 675
Cropped image 16 to: 36, 43, 716, 575
Cropped image 17 to: 6, 95, 781, 766
Cropped image 18 to: 49, 212, 654, 950
Cropped image 19 to: 106, 70, 752, 325
Cropped image 20 to: 65, 134, 801, 423
Cropped image 21 to: 49, 70, 883, 1119
Cropped image 22 to: 51, 80, 399, 352
Cropped image 23 to: 29, 47, 884, 1027
Cropped image 24 to: 16, 28, 765, 900
Cropped image 25 to: 37, 75, 584, 549
Cropped ima

With batch processing

In [9]:
def process_batch(batch_images, start_idx):
    batch_results = []
    
    for local_idx, image in enumerate(batch_images):
        idx = start_idx + local_idx
        
        # Convert to RGB
        if image.mode != 'RGB':
            image = image.convert('RGB')
        
        img_array = np.array(image)
        
        # Find most common color
        pixels = img_array.reshape(-1, 3)
        pixels_tuple = [tuple(p) for p in pixels]
        color_counts = Counter(pixels_tuple)
        most_common_color = color_counts.most_common(1)[0][0]
        
        batch_results.append((idx, image, img_array, most_common_color))
    
    # Process all images in batch on GPU
    for idx, image, img_array, most_common_color in batch_results:
        img_tensor = torch.from_numpy(img_array).to(device).float()
        
        bg_color1 = torch.tensor(most_common_color, device=device, dtype=torch.float32)
        mask1 = (img_tensor == bg_color1).all(dim=2)
        
        white_color = torch.tensor([255.0, 255.0, 255.0], device=device)
        distance_to_white = torch.norm(img_tensor - white_color, dim=2)
        mask2 = distance_to_white <= 10.0
        
        non_bg_mask = ~(mask1 | mask2)
        
        if non_bg_mask.any():
            rows = non_bg_mask.any(dim=1)
            cols = non_bg_mask.any(dim=0)
            
            top = rows.nonzero()[0].item()
            bottom = rows.nonzero()[-1].item() + 1
            left = cols.nonzero()[0].item()
            right = cols.nonzero()[-1].item() + 1
            
            height, width = img_array.shape[:2]
            padding = 10
            top = max(0, top - padding)
            bottom = min(height, bottom + padding)
            left = max(0, left - padding)
            right = min(width, right + padding)
            
            cropped_array = img_array[top:bottom, left:right]
            cropped_img = Image.fromarray(cropped_array)
            output_path = f"{output_dest_path}/image_{idx:04d}.png"
            cropped_img.save(output_path)
            print(f"Cropped image {idx} to: {left}, {top}, {right}, {bottom}")

# Process in batches
batch_size = 8
for i in range(0, len(image_list), batch_size):
    batch = image_list[i:i+batch_size]
    process_batch(batch, i)

Cropped image 0 to: 134, 71, 668, 530
Cropped image 1 to: 0, 45, 723, 902
Cropped image 2 to: 8, 0, 639, 618
Cropped image 3 to: 41, 26, 1027, 945
Cropped image 4 to: 134, 45, 1059, 1199
Cropped image 5 to: 191, 13, 570, 1040
Cropped image 6 to: 4, 38, 942, 1297
Cropped image 7 to: 39, 74, 835, 397
Cropped image 8 to: 56, 17, 954, 924
Cropped image 9 to: 0, 27, 987, 683
Cropped image 10 to: 46, 45, 878, 753
Cropped image 11 to: 66, 99, 435, 525
Cropped image 12 to: 12, 56, 943, 1197
Cropped image 13 to: 7, 63, 1080, 775
Cropped image 14 to: 0, 36, 1060, 837
Cropped image 15 to: 0, 44, 1044, 682
Cropped image 16 to: 3, 43, 1080, 556
Cropped image 17 to: 39, 62, 1011, 1127
Cropped image 18 to: 35, 46, 660, 681
Cropped image 19 to: 26, 98, 759, 798
Cropped image 20 to: 23, 50, 1080, 1239
Cropped image 21 to: 0, 38, 1080, 759
Cropped image 22 to: 36, 13, 551, 655
Cropped image 23 to: 3, 19, 1076, 1461
Cropped image 24 to: 22, 28, 1080, 1008
Cropped image 25 to: 39, 37, 1080, 1028
Cropped i